In [37]:
import numpy as np
import pandas as pd

In [38]:
from sklearn.impute import SimpleImputer # used to automatically fill missing values like None or NaN
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

# by default, SimpleImputer uses strategy='mean' - it is best for normally distributed numerical data
# 'median' is best for numerical data with extreme outliers
# 'mode' is best for categorical data

In [39]:
df = pd.read_csv('covid_toy.csv')

In [40]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [41]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [42]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],test_size=0.2)

In [43]:
X_train

,age,gender,fever,cough,city
67,65,Male,99.0,Mild,Bangalore
10,75,Female,NaN,Mild,Delhi
25,23,Male,NaN,Mild,Mumbai
13,64,Male,102.0,Mild,Bangalore
69,73,Female,103.0,Mild,Delhi
...,...,...,...,...,...
94,79,Male,NaN,Strong,Kolkata
52,47,Female,100.0,Strong,Bangalore
26,19,Female,100.0,Mild,Kolkata
29,34,Female,NaN,Strong,Mumbai


## 1. Aam Zindagi

In [44]:
# adding simple imputer to fever column
si = SimpleImputer()

X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever = si.transform(X_test[['fever']])

# passed inside double-brackets because all sklearn transformers and models expect input data in matrix form having shape (n_samples, n_features)

X_train_fever.shape

(80, 1)

In [45]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data
X_test_cough = oe.transform(X_test[['cough']])

# passed inside double-brackets because all sklearn transformers and models expect input data in matrix form having shape (n_samples, n_features)

X_train_cough.shape

(80, 1)

In [46]:
# OneHotEncoding -> gender,city
ohe = OneHotEncoder(drop='first',sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

# also the test data
X_test_gender_city = ohe.transform(X_test[['gender','city']])

# passed inside double-brackets because all sklearn transformers and models expect input data in matrix form having shape (n_samples, n_features)

X_train_gender_city.shape

(80, 4)

In [47]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape

(80, 1)

In [48]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)

# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

print(X_train_transformed.shape)
print(X_train)

(80, 7)
    age  gender  fever   cough       city
67   65    Male   99.0    Mild  Bangalore
10   75  Female    NaN    Mild      Delhi
25   23    Male    NaN    Mild     Mumbai
13   64    Male  102.0    Mild  Bangalore
69   73  Female  103.0    Mild      Delhi
..  ...     ...    ...     ...        ...
94   79    Male    NaN  Strong    Kolkata
52   47  Female  100.0  Strong  Bangalore
26   19  Female  100.0    Mild    Kolkata
29   34  Female    NaN  Strong     Mumbai
44   20    Male  102.0  Strong      Delhi

[80 rows x 5 columns]


## Mentos Zindagi

In [49]:
from sklearn.compose import ColumnTransformer

In [50]:
transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
    ],
    remainder='passthrough')

# remainder='passthrough' keeps the remaining columns without changes and remainder='drop' drops those columns which haven't been transformed

In [51]:
transformer.fit_transform(X_train).shape

(80, 7)

In [52]:
transformer.transform(X_test).shape

(20, 7)

In [56]:
transformer.get_feature_names_out()

array(['tnf1__fever', 'tnf2__cough', 'tnf3__gender_Male',
       'tnf3__city_Delhi', 'tnf3__city_Kolkata', 'tnf3__city_Mumbai',
       'remainder__age'], dtype=object)

In [58]:
# Use to display the whole transformed table

df_view = pd.DataFrame(transformer.fit_transform(X_train),
            columns = transformer.get_feature_names_out()
          )

In [59]:
df_view.sample(7)

,tnf1__fever,tnf2__cough,tnf3__gender_Male,tnf3__city_Delhi,tnf3__city_Kolkata,tnf3__city_Mumbai,remainder__age
55,100.0,1.0,0.0,0.0,1.0,0.0,11.0
10,102.0,0.0,0.0,1.0,0.0,0.0,49.0
11,101.0,1.0,0.0,0.0,1.0,0.0,51.0
41,100.0,0.0,1.0,0.0,1.0,0.0,55.0
38,104.0,1.0,0.0,1.0,0.0,0.0,34.0
67,98.0,0.0,1.0,0.0,1.0,0.0,24.0
70,101.0,0.0,0.0,0.0,0.0,1.0,65.0
